# 2.1. Multi-file Gathering và Dựng Cây Phân cấp Hierarchy

## 1. Phân giải đa ứng viên Compilation Root và mở rộng `\input`/`\include` chống chu trình

- **Khử nhập nhằng đa ứng viên Root (`gather.py`)**: Sau khi bóc tách comment `%` có bảo vệ `\%`, `\verb|...|` và các môi trường nguyên văn (`verbatim`, `lstlisting`, `minted`), trình quét tìm các tệp chứa `\documentclass`/`\documentstyle`. Khi một gói nguồn có nhiều tệp ứng viên (ví dụ kèm `supplement.tex`, `rebuttal.tex`), hệ thống phân giải tuần tự: ưu tiên cấu hình kiểm chứng [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json) $\rightarrow$ ứng viên duy nhất $\rightarrow$ **đối sánh độ phủ token tiêu đề `\title{...}` với metadata M1** (cứu thành công **`1.706` phiên bản đa tệp** trên tập 25.000 bài!).
- **Quy tắc dựng cây phân cấp & trạng thái `skip_level` (`hierarchy.py`)**: Gốc cây là `document`, đi xuống `chapter`/`section`/`subsection`..., `paragraph`, `list` $\rightarrow$ `item`, và 3 loại nút lá nội dung là `sentence`, `block_formula`, `figure` (hợp nhất cả `table`/`table*`/`figure*`). Khi gặp đề mục `References`/`Bibliography`, máy trạng thái bật `skip_level` để loại bỏ phần tài liệu tham khảo khỏi cây nội dung, nhưng **tự động khôi phục** nếu phía sau xuất hiện đề mục ngang cấp hoặc `\appendix` (bảo toàn trọn vẹn `Acknowledgements` và `Appendix`).

## 2. Thực thi trực tiếp Gathering + Vẽ Cây AST từ mã nguồn thật

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
import tempfile
from collections import defaultdict
from parser.gather import gather
from parser.hierarchy import parse_hierarchy

main_tex = r"""\documentclass{article}
\title{Demo Multi-file Paper}
\begin{document}
% \input{ignored_comment}
\input{sections/intro}
\section*{Acknowledgements}
We thank the team.
\appendix
\section{Appendix Proof}
Extra proof detail.
\section*{References}
\begin{thebibliography}{9}
\bibitem{k1} A. Author. Title. 2020.
\end{thebibliography}
\end{document}"""

intro_tex = r"""\section{Introduction}
First sentence with Fig. 1 and value 3.14.
\begin{itemize}\item First point.\item Second point.\end{itemize}
\begin{align}x + y = z\end{align}
\begin{table}[htpb]\centering\begin{tabular}{cc}\toprule A & B \\ \midrule 1 & 2 \\ \bottomrule\end{tabular}\end{table}
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp); (root / "sections").mkdir()
    (root / "main.tex").write_text(main_tex, encoding="utf-8")
    (root / "sections/intro.tex").write_text(intro_tex, encoding="utf-8")
    (root / "redundant.tex").write_text("Ignored standalone file.", encoding="utf-8")
    g = gather(root)
    tree = parse_hierarchy(g.text)

print(f"Compilation root: {g.root} ({g.root_selection}) | Đường dẫn biên dịch: {g.files} | Diagnostics: {g.diagnostics + tree.diagnostics}")
df_ast = pd.DataFrame([
    {"node_id": n.node_id, "parent_id": -1 if n.parent_id is None else n.parent_id, "kind": n.kind, "content": n.content[:85]}
    for n in tree.nodes
])
display(style_table(df_ast, caption="Bảng 6.1. Các nút cây phân cấp AST trích xuất trực tiếp từ bài báo đa tệp mẫu (loại bỏ redundant.tex và References).", int_cols=["node_id", "parent_id"]))

children = defaultdict(list)
by_id = {n.node_id: n for n in tree.nodes}
for n in tree.nodes:
    if n.parent_id is not None:
        children[n.parent_id].append(n.node_id)

pos, leaf_cursor = {}, [0]
def _layout_ast(nid: int, depth: int = 0) -> float:
    ch = children.get(nid, [])
    if not ch:
        y = -float(leaf_cursor[0]); leaf_cursor[0] += 1
    else:
        ys = [_layout_ast(c, depth + 1) for c in ch]
        y = sum(ys) / len(ys)
    pos[nid] = (depth, y)
    return y
_layout_ast(0)

palette = {
    "document": "#0F4C81", "section": "#0284C7", "paragraph": "#475569",
    "list": "#0D9488", "item": "#059669",
    "sentence": "#2563EB", "block_formula": "#D97706", "figure": "#B31B1B",
}
fig, ax = plt.subplots(figsize=(10.5, 5.5))
x_step = 2.35
for nid, (d, y) in pos.items():
    x = d * x_step
    for cid in children.get(nid, []):
        cd, cy = pos[cid]
        ax.plot([x + 1.88, cd * x_step - 0.04], [y, cy], color="#94A3B8", lw=1.25, zorder=1)
    n = by_id[nid]
    snippet = n.content.replace("\n", " ").strip()
    snippet = (snippet[:24] + "…") if len(snippet) > 25 else snippet
    label = f"[{n.node_id}] {n.kind}\n{snippet}"
    color = palette.get(n.kind, "#475569")
    ax.add_patch(FancyBboxPatch((x, y - 0.36), 1.86, 0.72, boxstyle="round,pad=0.05",
                                facecolor=color, edgecolor="none", alpha=0.94, zorder=2))
    ax.text(x + 0.93, y, label, ha="center", va="center", color="white", fontsize=7.5, fontweight="bold", zorder=3)

max_d = max(d for d, _ in pos.values())
ax.set_xlim(-0.15, (max_d + 1) * x_step)
ax.set_ylim(-leaf_cursor[0] + 0.2, 0.8)
ax.axis("off")
ax.set_title("Hình 6.1. Sơ đồ cây phân cấp AST sinh tự động từ `tree.nodes` (giữ Acknowledgements & Appendix, loại References)", pad=10)
plt.tight_layout(); plt.show()

## 3. Thống kê Gathering, Pilot nâng cấp (`v1 → v2`) và Phân bố 15.006.000 Phần tử

In [ ]:
gs = m2["gathering_stats"]
rs_df = pd.DataFrame([
    {"Cơ chế chọn Compilation Root": k, "Số phiên bản TeX": v, "Tỷ lệ / 38.554 TeX (%)": 100.0 * v / m2["versions_with_tex"]}
    for k, v in gs["root_selection"].items()
])
display(style_table(
    rs_df,
    caption=f"Bảng 6.2. Thống kê phân giải Compilation Root trên 38.554 phiên bản TeX (đa tệp tối đa {gs['max_comp_files']} tệp .tex trong 1 phiên bản).",
    int_cols=["Số phiên bản TeX"],
    pct_cols=["Tỷ lệ / 38.554 TeX (%)"],
    bar_cols={"Số phiên bản TeX": ("var(--ah-bar-fill)", None)},
))

pv = m2["pilot_v1_vs_v2"]
all_st_keys = ["success", "partial", "empty_no_tex", "error", "error_missing_source"]
comp = pd.DataFrame([
    {
        "Trạng thái": k,
        "Pilot v1 (trước vá)": int(pv["v1"]["status_counts"].get(k, 0)),
        "Pilot v2 (sau nâng cấp)": int(pv["v2"]["status_counts"].get(k, 0)),
        "Chênh lệch": int(pv["v2"]["status_counts"].get(k, 0)) - int(pv["v1"]["status_counts"].get(k, 0)),
    }
    for k in all_st_keys
])
display(style_table(
    comp,
    caption="Bảng 6.3. So sánh Pilot phân tầng 250 bài / 536 phiên bản trên Colab CPU trước và sau khi nâng cấp parser (giảm 80,0% lỗi!).",
    int_cols=["Pilot v1 (trước vá)", "Pilot v2 (sau nâng cấp)", "Chênh lệch"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.0))
xp = np.arange(len(comp)); wp = 0.36
ax1.bar(xp - wp/2, comp["Pilot v1 (trước vá)"], wp, color=BOOK_PALETTE["slate"], label="Pilot v1 (72,6% success)")
ax1.bar(xp + wp/2, comp["Pilot v2 (sau nâng cấp)"], wp, color=BOOK_PALETTE["emerald"], label="Pilot v2 (83,9% success)")
ax1.set_xticks(xp); ax1.set_xticklabels(["Success", "Partial", "Empty", "Error", "404"], rotation=0)
ax1.set_ylim(0, 470); ax1.grid(axis="y"); ax1.legend(fontsize=8.2)
ax1.set_ylabel("Số phiên bản"); ax1.set_title("Hình 6.2a. Tác động nâng cấp trên Pilot 250 bài (Lỗi: 70 → 14)")
for idx_c, r_c in comp.iterrows():
    ax1.text(idx_c + wp/2, r_c["Pilot v2 (sau nâng cấp)"] + 8, str(r_c["Pilot v2 (sau nâng cấp)"]),
             ha="center", fontsize=7.9, fontweight="bold")

diag_s = pd.Series(m2["diagnostic_categories"]).sort_values(ascending=True)
ax2.barh(diag_s.index, diag_s.values, color=BOOK_PALETTE["amber"], height=0.58)
ax2.set_xlim(0, diag_s.max() * 1.24); ax2.grid(axis="x")
ax2.set_xlabel("Số phiên bản có cảnh báo"); ax2.set_title("Hình 6.2b. Nhóm diagnostic trên 41.498 phiên bản")
for idx_d, v_d in enumerate(diag_s.values):
    ax2.text(v_d + 90, idx_d, fmt_int(v_d), va="center", fontsize=8.0, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
if "element_kind_counts" in m2:
    ek = pd.Series(m2["element_kind_counts"])
    df_ek = pd.DataFrame({
        "Loại nút cấu trúc (kind)": ek.index,
        "Số phần tử duy nhất": ek.values,
        "Tỷ trọng (%)": 100.0 * ek.values / ek.sum(),
    })
    display(style_table(
        df_ek,
        caption="Bảng 6.4. Phân bố 13 loại nút cấu trúc (`kind`) trên toàn bộ 15.006.000 phần tử của bảng `elements`.",
        int_cols=["Số phần tử duy nhất"],
        pct_cols=["Tỷ trọng (%)"],
        bar_cols={"Số phần tử duy nhất": ("var(--ah-bar-fill)", None)},
    ))
    fig, ax = plt.subplots(figsize=(9.8, 4.1))
    ek_sorted = df_ek.sort_values("Số phần tử duy nhất", ascending=True)
    ax.barh(ek_sorted["Loại nút cấu trúc (kind)"], ek_sorted["Số phần tử duy nhất"], color=BOOK_PALETTE["navy"], height=0.60)
    ax.set_xscale("symlog", linthresh=100); ax.set_xlim(0, 3e7); ax.grid(axis="x")
    ax.set_xlabel("Số phần tử duy nhất sau khử trùng (thang symlog)")
    ax.set_title("Hình 6.3. Phân bố 13 loại phần tử cấu trúc trên 15.006.000 nút bảng `elements`")
    for i_k, r_k in enumerate(ek_sorted.to_dict("records")):
        ax.text(r_k["Số phần tử duy nhất"] * 1.15 + 2, i_k,
                f"{fmt_int(r_k['Số phần tử duy nhất'])} ({fmt_pct(r_k['Tỷ trọng (%)'])})",
                va="center", fontsize=8.0, fontweight="bold")
    plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/gather.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/gather.py).
- Mã nguồn triển khai: [`src/parser/latex.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/latex.py).
- Mã nguồn triển khai: [`src/parser/hierarchy.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/hierarchy.py).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`configs/parser_roots.json`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/configs/parser_roots.json).